In [13]:

import os
import json
import gradio as gr
from dotenv import load_dotenv
import openai
from datetime import datetime
import sqlite3

# Load environment variables from .env file
load_dotenv(override=True)


True

In [14]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openai.api_key = openrouter_api_key

In [ ]:
import os
from openai import OpenAI

# Read runtime configuration from environment (fill .env or system env)
CHAT_MODEL = os.getenv("CHAT_MODEL", "meta-llama/llama-3.3-70b-instruct:free")  # free OpenRouter model
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "")  # speech->text model name or leave empty to use local Whisper
TTS_MODEL = os.getenv("TTS_MODEL", "")  # text->speech model name (OpenRouter/community) or local engine
TTS_VOICE = os.getenv("TTS_VOICE", "")  # voice name for your TTS engine
DB = os.getenv("DB", "voicemate.db")

# Note: keep your API keys out of source. Set OPENROUTER_API_KEY in .env or system env.
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [16]:
def init_db():
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            CREATE TABLE IF NOT EXISTS responses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                amount REAL NOT NULL,
                category TEXT NOT NULL,
                description TEXT,
                date TEXT NOT NULL,
                created_at TEXT NOT NULL,
            )
        ''')

        c.execute('''
            CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                due_date TEXT ,
                priority TEXT DEFAULT 'Medium',
                status TEXT DEFAULT 'Pending',
                created_at TEXT NOT NULL,
                completed_at
            )
        ''')
        c.execute('''
            CREATE TABLE IF NOT EXISTS journals (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                context TEXT NOT NULL,
                mood TEXT,
                date TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        ''')
        conn.commit()

In [ ]:
## Expense tools

def log_expenses(amount,category,description,date=None):
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    created_at = datetime.now().isoformat()
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            INSERT INTO responses (amount, category, description, date, created_at)
            VALUES (?, ?, ?, ?, ?)
        ''', (amount, category, description, date, datetime.now().isoformat()))
        conn.commit()

    return f"Logged expense: ${amount:.2f} for {category} on {date}. Description: {description}"

In [ ]:
def get_expense_summary():
    current_month = datetime.now().strftime("%Y-%m")
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            SELECT category, SUM(amount) 
            FROM expenses 
            WHERE strftime('%Y-%m', date) = ? 
            GROUP BY category
            ORDER BY SUM(amount) DESC''',
            (current_month,))
        rows = c.fetchall()
        conn.close()

        if not rows:
            return "No expenses logged for this month."
        
        total = 0
        for category, amount in rows:
            total += amount

        summary = f"Expense Summary for {current_month}:\n"
        for category, amount in rows:
            summary += f"- {category}: ${amount:.2f}\n"
        
        return summary + f"Total: ${total:.2f}"

In [ ]:
# Task management tools
def add_task(title, due_date=None, priority="Medium"):
    created_at = datetime.now().isoformat()
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            INSERT INTO tasks (title, due_date, priority,status, created_at)
            VALUES (?, ?, ?, ?, ?)
        ''', (title, due_date, priority, "Pending", created_at))
        conn.commit()
        task_id = c.lastrowid
        conn.close()

        if due_date:
            return f"Added task #{task_id}: '{title}' with due date {due_date} and priority {priority}."
        else:
            return f"Added task #{task_id}: '{title}' with priority {priority}."

def complete_task(task_id):
    completed_at = datetime.now().isoformat()
    try:
        with sqlite3.connect(DB) as conn:
            c = conn.cursor()
            c.execute('''
                SELECT id,title FROM tasks WHERE id = ? AND status = ?
            ''', (task_id, "Pending"))

    except:
        c.execute('SELECT id FROM tasks WHERE title LIKE ? AND status = ? LIMIT 1', (f"%{task_id}%", "Pending"))
        row = c.fetchone()

        if row is None:
            conn.close()
            return f"No pending task found with ID or title containing '{task_id}'."
        conn.commit()
        task_id = row[0]
        title = row[1]

        finished_time =datetime.now().isoformat()
        c.execute('''UPDATE tasks SET status = ?, completed_at = ? WHERE id = ?''', ("Completed", finished_time, task_id))
        conn.commit()
        conn.close()
        return f"Marked task #{task_id} ('{title}') as completed."  

   

def list_tasks(status="Pending"):
    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            SELECT id, title, due_date, priority 
            FROM tasks 
            WHERE status = ? 
            ORDER BY 
                CASE priority 
                    WHEN 'High' THEN 1 
                    WHEN 'Medium' THEN 2 
                    WHEN 'Low' THEN 3 
                    ELSE 4 
                END,
                due_date IS NULL, due_date
        ''', (status,))
        rows = c.fetchall()
        conn.close()

        if not rows:
            return f"No {status.lower()} tasks found."
        
        for row in rows:
            task_id = row[0]
            title = row[1]
            due_date = row[2] if row[2] else "No due date"
            priority = row[3]

        
        response = f"{status} Tasks:\n"
        for task_id, title, due_date, priority in rows:
            response += f"- #{task_id}: '{title}' (Priority: {priority}, Due: {due_date})\n"
        
        return response

In [ ]:
### Journal Tools

def add_journal_entry(context, mood=None, date=None):
    today = datetime.now().strftime("%Y-%m-%d")
    created_at = datetime.now().isoformat()

    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            INSERT INTO journals (context, mood, date, created_at)
            VALUES (?, ?, ?, ?)
        ''', (context, mood, date, created_at))
        conn.commit()
        conn.close()

        if mood:
            return "Journal entry added for {today}. Mood: {mood}"
        else:
            return "Journal entry added for {today}."

    

def get_journal_entries(days_back=7):

    how_far_back = datetime.timedelta(days=days_back)
    cutoff_datetime = datetime.now() - how_far_back
    cutoff_date = cutoff_datetime.strftime("%Y-%m-%d")
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()

    with sqlite3.connect(DB) as conn:
        c = conn.cursor()
        c.execute('''
            SELECT date, mood, context 
            FROM journals 
            WHERE created_at >= datetime('now', '-{days_back} days')
            ORDER BY created_at DESC
        ''', (days_back,))
        rows = c.fetchall()
        conn.close()

        if not rows:
            return f"No journal entries found. In the last {days_back} days."
        
        response = "Journal Entries:\n"
        for date, mood, context in rows:
            response += f"- {date} (Mood: {mood if mood else 'Not specified'}): {context}\n"
        
        for row in rows:
            date = row[0]
            mood = row[1] if row[1] else "Not specified"
            context = row[2]
            
            if len(context) > 120:
                snippet = context[:120] + "..."
            else:
                snippet = context

            if mood:
                response += f"- {date} (Mood: {mood}): {snippet}\n"
            else:
                response += f"- {date}: {snippet}\n"
        return response
                

In [ ]:
def get_daily_summary():
    today = datetime.now().strftime("%Y-%m-%d") 
    conn = sqlite3.connect(DB)
    c = conn.cursor()
    
    c.execute('SELECT COUNT(*), COALESCE(SUM(amount), 0) FROM responses WHERE date = ?''', (today,))
    expense_row = c.fetchone()
    expense_count = expense_row[0]
    expense_total = expense_row[1]

    c.execute('SELECT COUNT(*) FROM tasks WHERE status = ?',
              ("Pending",))

    pending_count = c.fetchone()[0]

    c.execute('SELECT COUNT(*) FROM tasks WHERE date = ? ORDER BY created_at DESC LIMIT 1', (today,))
    journal_row = c.fetchone()
    conn.close()

    response = f"Daily Summary for {today}:\n"
    response += f"- Expenses logged: {expense_count} (Total: ${expense_total:.2f})\n"
    response += f"- Pending tasks: {pending_count}\n"
    response += f"- Journal entries: {journal_row[0]}\n"

    if journal_row is not None:
        mood = journal_row[0]
        if mood is None:
            mood = "Not specified"
            response += f"  - Latest mood: {mood}\n"
        else:
            response += f"  - No entry yet today {today}\n"
    return response

In [ ]:
print("Initializing database...")
init_db()
print("Database initialized.")
print(log_expenses())
print(get_expense_summary())
print(add_task("Finish AI project", due_date="2024-07-01", priority="High"))    


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "log_expenses",
            "description": "Log an expense with amount, category, description, and optional date. Called when user calls for any spending, buying, or paying for anything.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount spent in rupees"},
                    "category": {"type": "string", "description": "Category of the expense (e.g. Food, Transport, Entertainment)"},
                    "description": {"type": "string", "description": "Brief description of the expense"},
                    "date": {"type": "string", "description": "Date of the expense in YYYY-MM-DD format (optional, defaults to today)"}
                },
                "required": ["amount", "category", "description"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": "Get a summary of expenses for the current month, grouped by category.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_task",
            "description": "Add a new task with title, optional due date, and priority.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Title of the task"},
                    "due_date": {"type": "string", "description": "Due date in YYYY-MM-DD format (optional)"},
                    "priority": {"type": "string", "description": "Priority (High, Medium, Low). Defaults to Medium."}
                },
                "required": ["title"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "complete_task",
            "description": "Mark a task as completed by ID or title.",
            "parameters": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string", "description": "Task ID or part of the title"}
                },
                "required": ["task_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_tasks",
            "description": "List all tasks with a given status (Pending or Completed).",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {"type": "string", "description": "Task status to filter by (Pending or Completed). Defaults to Pending."}
                },
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_journal_entry",
            "description": "Add a journal entry with context, optional mood, and optional date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {"type": "string", "description": "Journal entry text"},
                    "mood": {"type": "string", "description": "Mood (optional)"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format (optional, defaults to today)"}
                },
                "required": ["context"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_journal_entries",
            "description": "Get journal entries from the last N days (default 7).",
            "parameters": {
                "type": "object",
                "properties": {
                    "days_back": {"type": "integer", "description": "Number of days back to fetch entries (optional, default 7)"}
                },
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_daily_summary",
            "description": "Get a summary of today's expenses, tasks, and journal entries.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
#tool call loop
available_tools = {
    "log_expenses": log_expenses,
    "get_expense_summary": get_expense_summary,
    "add_task": add_task,
    "complete_task": complete_task,
    "list_tasks": list_tasks,
    "add_journal_entry": add_journal_entry,
    "get_journal_entries": get_journal_entries,
    "get_daily_summary": get_daily_summary,
}

In [ ]:
def chat(message,history):
    today_str = datetime.now().strftime("%Y-%m-%d")
    day_name = datetime.now().strftime("%A")
    response = f'''
    You are Voicemate, a personal assistant designed to help me manage my daily life through voice interactions. Today is {day_name}, {today_str}. I can assist with logging expenses, managing tasks, and keeping a journal. I can also provide summaries of your activities and expenses. Please let me know how I can assist you today!  
    Ony Speak in English and use the provided tools to perform actions when needed. If the user asks for something that can be done with a tool, call the appropriate tool with the correct parameters. Always use the tools when relevant instead of providing information directly.
    Today is {today_str} and {day_name}.tools
    Rules - follow strictly
    -user mentions spending -> call log_expenses with amount, category, description, and optional date.
    -user asks about expenses -> call get_expense_summary to provide a summary of expenses for the current month.
    -user wants to add a task -> call add_task with title, optional due date, and priority.
    -user wants to complete a task -> call complete_task with task ID or title.
    -user wants to list tasks -> call list_tasks with optional status filter (Pending or Completed).
    -user wants to add a journal entry -> call add_journal_entry with context, optional mood, and optional date.
    -user wants to see journal entries -> call get_journal_entries with optional days_back parameter.
    -user wants a daily summary -> call get_daily_summary to provide a summary of today's activities.
    -Always respond in a conversational manner, but use the tools to perform actions and fetch information instead of providing it directly.
    Keep responses short and human. Confirm action and not displaying raw data. convert days to yyyy-mm--dd before tool calling.tools
    '''

    messages = [{"role": "system", "content": response}]
    for h in history:
        messages.append({"role": "['role']", "content": h['content']})
        messages.append({"role": "user", "content": message})

        response = openai.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools = tools)
        
        while response.choices[0].finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            responses = handle_tool_calls(message_obj)
            messages.append({"role": "assistant", "content": message_obj.content})
            message.extend(responses)
            response = openai.chat.completions.create(
                model=CHAT_MODEL,
                messages=messages,
                tools = tools)
            
        return response.choices[0].message.content
            
            


In [ ]:
def transcribe_audio(audio_file):
    if WHISPER_MODEL:
        # Call OpenRouter Whisper API
        with open(audio_file, "rb") as f:
            response = openai.audio.transcriptions.create(
                file=f,
                model=WHISPER_MODEL,
                language="en"
            )
        return response.text

In [ ]:
def text_to_speech(text):
    if TTS_MODEL:
        # Call OpenRouter TTS API
        response = openai.audio.speech.create(
            model=TTS_MODEL,
            voice=TTS_VOICE,
            input=text
        )
        audio_url = response.url
        return audio_url

In [ ]:
#gradio ui